# Workshop 7a · Demo 1: Data Cleaning

**Author:** Nursen Aydin

In this demo we clean a small survey dataset step by step, following the *data quality audit* from our lecture. We use the **pandas** library.

**By the end you can:**
1. Check what "missing" really means in a file.
2. Fix inconsistent category labels.
3. Spot implausible values and decide what to do with them.
4. Handle missing values: delete, fill (with a missing flag), or keep as a category.
5. Set the correct data types.
6. Keep a **cleaning log**: every change, with a reason.

The file **survey.csv** contains answers from 237 students: gender, writing hand, pulse, how often they exercise, smoking, and height (cm).

**Before you start:** put `survey.csv` in the same folder as this notebook. *Using Google Colab?* Click the folder icon on the left, then the upload button, and upload `survey.csv` (Colab removes uploaded files when you close it, so upload the CSV again each time you open the notebook.).

-----
## 1. Load the data and take a first look

We start the way most tutorials do: `pd.read_csv()` with its default settings.

In [ ]:
# Import the pandas library and give it the short name pd
import pandas as pd

# Load the file. Make sure 'survey.csv' is in the same folder as this notebook
datafile = pd.read_csv(---)

# How many rows and columns? You can use 'shape' function
print(datafile.---)

In [ ]:
# The first five rows
datafile.---

In [ ]:
# The structure: column names, data types and the number of non-missing values
datafile.---

**What do you notice?**
- The first column, `Unnamed: 0`, is just a row number. It is stored as a number, but it is not a quantity. We will remove it.
- Gender, Handedness, Exercise and Smoke are stored as text (`object`, or `str` in newer versions of pandas). We will turn them into categories later.
- Some columns have fewer than 237 (non-null entries) non-missing values: there are missing values.

-----
## 2. Let's investigate: what does "missing" mean here?

Before we decide what to do with missing values, we check **what they are**. The method `value_counts(dropna=False)` counts every value in a column, *including* the missing ones (shown as `NaN`).

In [ ]:
# Count every answer to the Exercise question, including missing values
datafile["---"].value_counts(---)

The survey asked *how often do you exercise?* We see **Freq** and **Some**, and 24 missing values. It looks as if none of the 237 respondents said they do not exercise at all, which is unlikely in a group this size. Suspicious!

Let's look at the **raw file**, exactly as it is stored, before pandas interprets it.

In [ ]:
# Look at the file as plain text: no guessing what is missing
# keep_default_na=False: turn nothing into NaN

pd.read_csv("survey.csv", keep_default_na=False).head()

Two different things are going on here:
- **Blank cells** (Pulse in row 3, Height in row 2): nothing was recorded. These really are missing.
- The word **None** (Exercise): these students answered "I do not exercise". That is a real answer, but pandas treats the word `None` as *missing* by default, so 24 real answers disappeared.

**Fix:** tell pandas exactly which values mean "missing". Here, only an empty cell or the text `NA`:

```python
pd.read_csv("survey.csv", keep_default_na=False, na_values=["", "NA"])
```
- `keep_default_na=False`: do not guess.
- `na_values=["", "NA"]`: only these mean missing.

In [ ]:
# Load the file again, this time telling pandas what "missing" means
datafile = pd.read_csv("survey.csv", keep_default_na=---, na_values=---)

# Check the Exercise column again
datafile["Exercise"].value_counts(dropna=False)

Now **None** is back as a proper answer for 24 students.

**Why it matters:** if we had removed all rows with missing values, we would have silently deleted every student who does not exercise. Any analysis of exercise and pulse would then be wrong.

> **Lesson:** never trust that "missing" means missing. Check the raw data first. 

-----
## 3. Remove columns that carry no information

Some columns only identify rows, such as row numbers or customer IDs. They are stored as numbers, but they are not quantities. Here, `Unnamed: 0` just numbers the rows 1, 2, 3, …, so we remove it.

In [ ]:
# Remove the row-number column
datafile = datafile.drop(columns=[---])
datafile.head()

-----
## 4. Category labels

*Female*, *female* and *"Female "* (with a space at the end) look the same to us, but to Python they are *three different groups*. If we do not fix this, a machine learning model later treats them as different categories. So before using any text column, we check its labels.

In [ ]:
# Check the labels of every text column
for column in ["Gender", "Handedness", "Exercise", "Smoke"]:
    print(datafile[---].value_counts(), "\n")

Gender has four labels: **Female, female, Male, male**. We fix them with `.replace()`, which takes a dictionary `{old: new}`:

```python
df[column name] = df[column name].replace({old value: new value})
```

*Tip:* fix labels **before** converting a column to a category, otherwise the old labels remain as empty categories.

In [ ]:
# Replace female with Female and male with Male
datafile["---"] = datafile["---"].replace({"female": ---, "male": ---})

# Check again
print(datafile["---"].value_counts())

*Other label problems you may meet:* stray spaces (`"Male "`) are removed with `.str.strip()`; different capitalisation with `.str.title()` or `.str.lower()`.

-----
## 5. Implausible values

`describe()` gives a summary of the numerical columns. Look at the minimum and maximum: are they plausible?

In [ ]:
# Summary statistics of the numerical columns
datafile.---

In [ ]:
# Sort by Pulse to see the lowest values
datafile.sort_values(---).head()

**Discuss:**
- **Pulse 40:** an error, or a very fit athlete? A resting pulse of 40 is rare but possible. Without a way to check, we **keep** it and record the decision.

-----
## 6. Missing values

First, how many values are missing in each column, and what share?

In [ ]:
# Number of missing values per column: you can use isnull() and sum() functions
print(datafile.---)

# Share of missing values per column (%)
print((datafile.--- / len(datafile) * 100).round(1))

### Option A: delete every row with a missing value

`dropna()` removes every row that has at least one missing value.

In [ ]:
# Remove rows with any missing value
datafile_dropped = datafile.---

# How many rows did we lose?
rows_removed = datafile.shape[0] - datafile_dropped.shape[0]
print(f"Rows removed: {rows_removed} of {datafile.shape[0]} ({rows_removed / datafile.shape[0]:.0%})")

We would lose **more than a quarter** of the students, mostly because their pulse is missing. That is a lot of information to throw away.

### Option B: fill numbers with the median, and add a missing flag

Instead of deleting rows, we fill each gap with a typical value.

- **Median:** for numbers, we use the median. We prefer it to the mean because a few extreme values, such as a pulse of 40, do not pull it up or down.
- **Missing flag:** before filling, we add a new column that is **1** where the value was missing and **0** where it was observed. After filling, the gaps are no longer visible, so the flag is our record of which values we filled in. It also lets us check later whether the students with missing values were different from the others.

In [ ]:
# Add missing flags BEFORE filling (1 = was missing, 0 = observed)
datafile["Pulse_missing"] = datafile["Pulse"].isnull().astype(int)
datafile["Height_missing"] = datafile[---].isnull().astype(int)

# Medians
median_pulse = datafile["Pulse"].---
median_height = datafile["Height"].---
print("Median pulse:", median_pulse, " Median height:", median_height)

# Fill the missing values with the medians
datafile = datafile.fillna({"Pulse": ---, "Height": ---})
print(datafile[["Pulse", "Height"]].isnull().sum())

### Categories with missing values

Gender, Handedness and Smoke have one missing value each. Two common choices:
- the most frequent category (the **mode**), when only a few are missing;
- an explicit **"Missing"** category, when the gap itself may carry information.

In [ ]:
# One missing value each: fill with the most frequent category (the mode)
# (Gender is a tie, 118 Female and 118 Male: mode()[0] takes the first in alphabetical order. Worth a note in the log.)
for column in ["Gender", "Handedness", "Smoke"]:
    most_frequent = datafile[column].---
    datafile[column] = datafile[column].fillna(most_frequent)

# Check: no missing values left
print(datafile.isnull().sum())

*Note:* in a real modelling project, values such as the median are calculated on the *training set only*, after splitting the data, and then used to fill both the training and the test set. Otherwise the test set has helped to clean the data, and the test result looks slightly better than it should. Here we fill before splitting to keep things simple and also we are not using this dataset to build a model.

-----
## 7. Set the correct data types

Now that the labels are clean, we store the four text columns as **categories** with `astype("category")`.

*Demo 2:* some of these categories have a natural order (Exercise: None < Some < Freq; Smoke: Never < … < Heavy). In Demo 2 we will see how to use such an order when we **encode** categories.

In [ ]:
# The four text columns
category_columns = [---]

# Store them as categories
datafile[category_columns] = datafile[category_columns].astype(---)

# Check the structure again
datafile.info()

-----
## 8. The cleaning log

Lecture 2: *record why each cleaning rule exists.* Every change we made, with a reason:

| Step | What we did | Why | Rows affected |
|---|---|---|---|
| Reading | Only empty cells and "NA" count as missing | "None" is a real answer to the Exercise question | 24 values recovered |
| ID column | Removed `Unnamed: 0` | A row number, not a quantity | all |
| Labels | female → Female, male → Male | Same category written differently | 7 |
| Implausible values | Kept Pulse 40 | Rare but possible; no source to check | 0 |
| Pulse, Height | Median fill + missing flag | Deleting would lose 29% of students | 45 and 28 values |
| Gender, Handedness, Smoke | Mode fill (Gender: a tie, Female chosen) | One missing value each | 3 values |
| Data types | Text columns stored as categories | Correct measurement levels | all |

In [ ]:
# Final check of the cleaned data
print(datafile.shape)
datafile.head()

-----
**Next:** in **Demo 2** we prepare data for modelling (encoding, splitting and balancing). Then it is your turn in the **Exercise**.